In [1]:
from tkinter import Image
from PIL import ImageFile
from ultralytics import YOLO
import cv2
import pandas as pds
import numpy as np
import os
from tensorflow.python.autograph.converters.break_statements import transform
from torch.utils.data import Dataset
import cv2
import matplotlib.pyplot as plt

In [52]:
#class KITII(Dataset):
    def __init__(self,image_dir,labels_dir,transform = None):
        self.image_dir = image_dir
        self.labels_dir= labels_dir
        self.transform = transform

        IMAGE_DIR = r'C:\Users\Mobina\PyCharmMiscProject\KITTI training images\image_2'
        LABEL_DIR = r'C:\Users\Mobina\PyCharmMiscProject\KITTI training images\label_2'

        self.image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(".png")])
        self.label_files = sorted([f for f in os.listdir(labels_dir) if f.endswith(".txt")])

        with open(IMAGE_DIR,'r') as f:
            pass
            image_list_path = f.readlines()
            #labels_list_path = f.readlines()
            labels_list_path =[(image.replace("image","label")
                                    .replace("png","txt")
                                for image in image_list_path)]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)


    def parse_label(selfself,lbl_path):
        object = []
        with open(lbl_path,'r') as f:
            for line in f.readlines():
                parts = line.split(" ")
                if parts[0] == "dont care":
                    continue
                object.append({
                    'class':parts[0],
                    'bbox': [float(parts[4]),float(parts[5]),
                            float(parts[6]),float(parts[7])]})



    def __getitem__(self,indx):
        img_name = self.image_files[indx]
        img_path = os.path.join(self.image_dir,img_name)
        image = Image.open(img_path).convert('RGB')

        lbl_name = self.image_name.replac('.png','.txt')
        lbs_path = os.path.join(self.labels_dir,lbl_name)
        objects = self.parse_label(lbs_path)
        if self.transform:
            img = self.transform(Image.open(imgpath).convert('RGB'))
            img, lbl = self.transform(img), self.transform(lblpath)
        return


In [3]:
import os
import cv2
import torch
from torch.utils.data import Dataset
from PIL import Image

class KITTI(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.transform = transform

        # لیست همه تصاویر
        self.image_files = sorted([
            f for f in os.listdir(image_dir) if f.endswith('.png')
        ])

    def __len__(self):
        return len(self.image_files)

    def parse_label(self, label_path):
        objects = []
        with open(label_path, 'r') as f:
            for line in f.readlines():
                parts = line.strip().split(' ')
                if parts[0] == 'DontCare':
                    continue
                objects.append({
                    'class': parts[0],
                    'bbox': [float(parts[4]), float(parts[5]),
                             float(parts[6]), float(parts[7])]
                })
        return objects

    def __getitem__(self, idx):
        # لود تصویر
        img_name = self.image_files[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        # لود label
        lbl_name = img_name.replace('.png', '.txt')
        lbl_path = os.path.join(self.label_dir, lbl_name)
        objects = self.parse_label(lbl_path)

        if self.transform:
            image = self.transform(image)

        return image, objects





In [4]:
img, objects = dataset[0]
print(f'Image size: {img.size}')
print(f'Objects in this image: {len(objects)}')
for obj in objects[:3]:
    print(f'  {obj["class"]}: {obj["bbox"]}')

NameError: name 'dataset' is not defined

In [5]:
model = YOLO('yolov8n.pt')

results = model(r'C:\Users\Mobina\PyCharmMiscProject\KITTI training images\image_2\000000.png')
results[0].show()


image 1/1 C:\Users\Mobina\PyCharmMiscProject\KITTI training images\image_2\000000.png: 224x640 1 person, 1 bicycle, 1 skateboard, 1 potted plant, 56.9ms
Speed: 2.9ms preprocess, 56.9ms inference, 3.4ms postprocess per image at shape (1, 3, 224, 640)


In [6]:
CLASSES = {'Car': 0, 'Pedestrian': 1, 'Cyclist': 2, 'Van': 3, 'Truck': 4}

IMAGE_DIR = r'C:\Users\Mobina\PyCharmMiscProject\KITTI training images\image_2'
LABEL_DIR = r'C:\Users\Mobina\PyCharmMiscProject\KITTI training images\label_2'
OUTPUT_DIR = r'C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\labels'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# اندازه تصاویر کیتی
IMG_W, IMG_H = 1242, 375

converted = 0
skipped = 0

for lbl_file in sorted(os.listdir(LABEL_DIR)):
    input_path = os.path.join(LABEL_DIR, lbl_file)
    output_path = os.path.join(OUTPUT_DIR, lbl_file)

    yolo_lines = []

    with open(input_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split(' ')
            class_name = parts[0]

            if class_name not in CLASSES:
                skipped += 1
                continue

            class_id = CLASSES[class_name]
            x1, y1, x2, y2 = float(parts[4]), float(parts[5]), float(parts[6]), float(parts[7])

            # تبدیل به فرمت یولو
            cx = (x1 + x2) / 2 / IMG_W
            cy = (y1 + y2) / 2 / IMG_H
            w  = (x2 - x1) / IMG_W
            h  = (y2 - y1) / IMG_H

            yolo_lines.append(f'{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

    with open(output_path, 'w') as f:
        f.write('\n'.join(yolo_lines))

    converted += 1

print(f'تبدیل شد: {converted} فایل')
print(f'رد شد: {skipped} آبجکت (کلاس‌های غیرمهم)')

تبدیل شد: 7481 فایل
رد شد: 13001 آبجکت (کلاس‌های غیرمهم)


In [9]:
path: C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo
train: images/train
val: images/val

nc: 5
names: ['Car', 'Pedestrian', 'Cyclist', 'Van', 'Truck']

SyntaxError: invalid syntax (3540941962.py, line 1)

In [11]:
import os
import shutil
import random

# مسیرها
IMAGE_DIR = r'C:\Users\Mobina\PyCharmMiscProject\KITTI training images\image_2'
YOLO_LABEL_DIR = r'C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\labels'
OUTPUT_BASE = r'C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo'

# ساخت پوشه‌ها
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_BASE, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_BASE, 'labels', split), exist_ok=True)

# لیست همه فایل‌ها و shuffle
all_files = sorted([f.replace('.txt', '') for f in os.listdir(YOLO_LABEL_DIR)
                    if f.endswith('.txt')])
random.seed(42)
random.shuffle(all_files)

# ۸۰٪ train، ۲۰٪ val
split_idx = int(len(all_files) * 0.8)
train_files = all_files[:split_idx]
val_files = all_files[split_idx:]

def copy_files(file_list, split):
    for name in file_list:
        # کپی تصویر
        src_img = os.path.join(IMAGE_DIR, name + '.png')
        dst_img = os.path.join(OUTPUT_BASE, 'images', split, name + '.png')
        shutil.copy(src_img, dst_img)

        # کپی لیبل
        src_lbl = os.path.join(YOLO_LABEL_DIR, name + '.txt')
        dst_lbl = os.path.join(OUTPUT_BASE, 'labels', split, name + '.txt')
        shutil.copy(src_lbl, dst_lbl)

copy_files(train_files, 'train')
copy_files(val_files, 'val')

print(f'Train: {len(train_files)} تصویر')
print(f'Val: {len(val_files)} تصویر')

Train: 5984 تصویر
Val: 1497 تصویر


In [10]:
all_files = sorted([f.replace('.txt', '') for f in os.listdir(YOLO_LABEL_DIR)])
print(f'تعداد: {len(all_files)}')
print(f'چند تا اول: {all_files[:5]}')
print(f'چند تا آخر: {all_files[-5:]}')

تعداد: 7483
چند تا اول: ['000000', '000001', '000002', '000003', '000004']
چند تا آخر: ['007478', '007479', '007480', 'train', 'val']


In [12]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=r'C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\kitti.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    name='kitti_yolov8',
    patience=10,
)


New https://pypi.org/project/ultralytics/8.4.66 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.64  Python-3.10.11 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\kitti.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, mom

In [15]:
from ultralytics import YOLO

# لود بهترین مدل
model = YOLO(r'C:\Users\Mobina\PyCharmMiscProject\runs\detect\kitti_yolov8\weights\best.pt')

# تست روی چند تصویر
test_images = [
    r'C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\images\val\000004.png',
    r'C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\images\val\000017.png',
    r'C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\images\val\000053.png',
]

for img_path in test_images:
    results = model(img_path, conf=0.5)
    results[0].show()



image 1/1 C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\images\val\000004.png: 224x640 2 Cars, 1 Van, 25.4ms
Speed: 4.5ms preprocess, 25.4ms inference, 2.1ms postprocess per image at shape (1, 3, 224, 640)

image 1/1 C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\images\val\000017.png: 224x640 1 Car, 24.1ms
Speed: 3.1ms preprocess, 24.1ms inference, 5.8ms postprocess per image at shape (1, 3, 224, 640)

image 1/1 C:\Users\Mobina\PyCharmMiscProject\KITTI_yolo\images\val\000053.png: 224x640 5 Cars, 1 Van, 23.0ms
Speed: 4.4ms preprocess, 23.0ms inference, 3.3ms postprocess per image at shape (1, 3, 224, 640)
